# EXIT chart

The equalizer's characteristic against the decoder's, mirrored. The gap between them is the tunnel the turbo loop climbs.

In [ ]:
import json, subprocess, pathlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent
MSPRS = ROOT / "build" / "bin" / "msprs"
EXIT = ROOT / "results" / "exit"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "legend.fontsize": 7, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.2,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

EB_N0_DB = 6.0


In [ ]:
def characteristic(name, snr):
    """One EXIT curve from results/exit, matched on value not key spelling."""
    d = json.loads((EXIT / f"{name}.json").read_text())
    key = min(d["results"], key=lambda s: abs(float(s) - snr))
    return np.array(d["IA"]), np.array(d["results"][key]["IE_avg"])

def decoder():
    d = json.loads((EXIT / "coder_K3.json").read_text())
    return np.array(d["IA"]), np.array(d["IE_avg"])

def staircase(ia, ie, dIA, dIE, iters=7):
    xs, ys, x = [], [], 0.0
    for _ in range(iters):
        y = float(np.interp(x, ia, ie))
        xs += [x, x]; ys += [ys[-1] if ys else 0.0, y]
        nx = float(np.interp(y, dIA, dIE))
        xs += [x, nx]; ys += [y, y]
        if nx - x < 1e-4:
            break
        x = nx
    return xs, ys

In [ ]:
dIA, dIE = decoder()
fams = [(f, characteristic(f"nsm_L3_{f}", EB_N0_DB)) for f in ("balanced", "unbalanced")]

fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.3), sharey=True)
for ax, (fam, (ia, ie)) in zip(axes, fams):
    ax.plot(dIE, dIA, color="#9467bd", ls="--", lw=1.5, label="Conv. decoder, $K=3$")
    ax.plot(ia, ie, color="#000000", lw=2.0, label="MS-PRS $L_0=3$")
    xs, ys = staircase(ia, ie, dIA, dIE)
    ax.plot(xs, ys, color="#000000", lw=1.2, alpha=0.85, ls=":")
    ax.plot([xs[-1]], [ys[-1]], "o", ms=5, color="#000000")
    ax.plot([0, 1], [0, 1], ls=":", lw=0.7, color="0.6")
    ax.set(xlim=(0, 1), ylim=(0, 1),
           xlabel=r"$I_A^{\mathrm{modem}} = I_E^{\mathrm{dec}}$",
           title=f"{fam}, rise {ie[-1] - ie[0]:+.2f}")

axes[0].set_ylabel(r"$I_E^{\mathrm{modem}} = I_A^{\mathrm{dec}}$")
axes[0].legend(loc="lower right")
fig.suptitle(f"$E_b/N_0$ = {EB_N0_DB:g} dB", y=0.99)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIGURES / f"exit_turbo_gain.{ext}")

for fam, (ia, ie) in fams:
    print(f"{fam:11s} IE(0)={ie[0]:.3f} IE(1)={ie[-1]:.3f} rise={ie[-1]-ie[0]:+.3f}")